# 1. Import Libraries

In [13]:
import pandas as pd
import numpy as np
from sklearn.cross_decomposition import PLSRegression

from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import root_mean_squared_error

import xgboost as xgb

# 2. Load Dataset

In [5]:
df_train = pd.read_excel("../Dataset/training.xlsx")
df_test = pd.read_excel("../Dataset/testing.xlsx")

In [21]:
ANCHOR_YEARS = {1: 2020, 2: 2021, 3: 2022}

In [22]:
df_train['Year'] = df_train['Year'].map(ANCHOR_YEARS)
df_test['Year'] = df_test['Year'].map(ANCHOR_YEARS)

df_train['Date'] = pd.to_datetime(df_train[['Year', 'Month', 'Day', 'Hour']])
df_test['Date'] = pd.to_datetime(df_test[['Year', 'Month', 'Day', 'Hour']])

# 3. Dimension Reducing using PLS

## 3.1 Find the best n_components using TimeSeriesSplit CV

In [ ]:
features_temp = ['Site-1 Temp', 'Site-2 Temp',
       'Site-3 Temp', 'Site-4 Temp', 'Site-5 Temp']

features_ghi = [
'Site-1 GHI', 'Site-2 GHI',
       'Site-3 GHI', 'Site-4 GHI', 'Site-5 GHI']

target = 'Load'

In [12]:
X_temp = df_train[features_temp]
X_ghi = df_train[features_ghi]
y = df_train[target]

In [19]:
tscv = TimeSeriesSplit(n_splits=4)
cv_scores = []
best_k = 1
best_rmse = float('inf')

for k in range(1,6):

    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_train)):
        train_fold = df_train.iloc[train_idx].copy()
        val_fold = df_train.iloc[val_idx].copy()

        pls_temp = PLSRegression(n_components=k)
        pls_temp.fit(X_temp, y)

        pls_ghi = PLSRegression(n_components=k)
        pls_ghi.fit(X_ghi, y)

        for comp in range(k):
            train_fold[f'Temp_PLS_{comp+1}'] = pls_temp.transform(train_fold[features_temp])[:, comp]
            train_fold[f'GHI_PLS_{comp+1}'] = pls_ghi.transform(train_fold[features_ghi])[:, comp]
            
            val_fold[f'Temp_PLS_{comp+1}'] = pls_temp.transform(val_fold[features_temp])[:, comp]
            val_fold[f'GHI_PLS_{comp+1}'] = pls_ghi.transform(val_fold[features_ghi])[:, comp]

        features = ['Year', 'Month', 'Day', 'Hour'] + \
                       [f'Temp_PLS_{i+1}' for i in range(k)] + \
                       [f'GHI_PLS_{i+1}' for i in range(k)]
                       
        model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
        model.fit(train_fold[features], train_fold['Load'])
        
        preds = model.predict(val_fold[features])
        cv_scores.append(root_mean_squared_error(val_fold['Load'], preds))

    mean_rmse = np.mean(cv_scores)
    print(f"n_components = {k} | Mean CV RMSE: {mean_rmse:.4f}")

    if mean_rmse < best_rmse:
        best_k = k
        best_rmse = mean_rmse

print(f"Best n_components: {best_k}")

n_components = 1 | Mean CV RMSE: 322.1366
n_components = 2 | Mean CV RMSE: 314.2459
n_components = 3 | Mean CV RMSE: 309.6097
n_components = 4 | Mean CV RMSE: 306.9336
n_components = 5 | Mean CV RMSE: 306.3163
Best n_components: 5


In [ ]:
def evaluate_pls_components(X, y, max_components=5):
    print("Component | R2 Score on Training Set")
    print("-" * 35)
    for k in range(1, max_components + 1):
        pls = PLSRegression(n_components=k)
        pls.fit(X, y)
        score = pls.score(X, y)
        print(f"   {k}      | {score:.4f}")

evaluate_pls_components(X_temp, y)
evaluate_pls_components(X_ghi, y)

Component | R2 Score on Training Set
-----------------------------------
   1      | 0.1636
   2      | 0.1650
   3      | 0.1658
   4      | 0.1658
   5      | 0.1658
Component | R2 Score on Training Set
-----------------------------------
   1      | 0.0034
   2      | 0.0063
   3      | 0.0065
   4      | 0.0066
   5      | 0.0066
